In [ ]:
'''
Compare e^{tL} with 1st order Euler integrator with different steps. 
(exact vs superexact)
'''
import numpy as np
import qutip as qt

def build_dissipative_tfim(N, J, h, gamma):
    # 1. 构建 Hamiltonian
    si_x = [
        qt.tensor([qt.qeye(2)] * i + [qt.sigmax()] + [qt.qeye(2)] * (N - i - 1))
        for i in range(N)
    ]
    si_z = [
        qt.tensor([qt.qeye(2)] * i + [qt.sigmaz()] + [qt.qeye(2)] * (N - i - 1))
        for i in range(N)
    ]

    H_x = 0
    H_z = 0
    # 相互作用项
    for i in range(N - 1):
        H_z += -J * si_z[i] * si_z[i + 1]
    # 横场项
    for i in range(N):
        H_x += -h * si_x[i]

    H = H_x + H_z

    # 2. 构建耗散算符 (Collapse Operators)
    # 这里使用 sigma_minus (自旋弛豫)
    c_ops = []
    for i in range(N):
        # sigmam() 是降算符
        L_i = qt.tensor([qt.qeye(2)] * i + [qt.sigmam()] + [qt.qeye(2)] * (N - i - 1))
        c_ops.append(np.sqrt(gamma) * L_i)  # why square root here

    return H_x, H_z, c_ops


for N in [3, 4, 5]:
    # N = 5
    gamma = 0.1

    H_x, H_z, c_ops = build_dissipative_tfim(N, J=1.0, h=0.5, gamma=gamma)

    psi = qt.tensor([qt.basis(2, 0) for _ in range(N)])

    # 转换为密度矩阵 rho = |psi><psi|
    rho_0 = qt.ket2dm(psi)

    # # 求解稳态 (Steady State)
    # rho_ss = qt.steadystate(H_x + H_z, c_ops)
    t = 0.2

    L = qt.liouvillian(H_x + H_z, c_ops)
    exact = (L * t).expm()

    # store the exact output density matrix
    rho_superexact = exact * qt.operator_to_vector(rho_0)
    rho_superexact = qt.vector_to_operator(rho_superexact)
    # np.save(
    #     f"./data_sigmam/|000>initial/gamma_{gamma}/superexact/rho_superexact_N_{N}.npy",
    #     rho_superexact.full(),
    # )

    # compare the 1st order integrator and e^{tL}
    step = 10000
    print('N =', N)
    rho_exact1 = np.load(
        f"./data_sigmam/|000>initial/gamma_{gamma}/exp(tL)_Euler/rho_exact_N_{N}_step_{step}.npy"
    )
    diff1 = rho_exact1 - rho_superexact.full()
    print(
        f"difference between exact e^tL and {step} steps 1st order update:\n",
        np.linalg.svd(diff1, compute_uv=False).sum(),
    )
    step = 100000
    rho_exact2 = np.load(
        f"./data_sigmam/|000>initial/gamma_{gamma}/exp(tL)_Euler/rho_exact_N_{N}_step_{step}.npy"
    )
    diff2 = rho_exact2 - rho_superexact.full()
    print(
        f"difference between exact e^tL and {step} steps 1st order update:\n",
        np.linalg.svd(diff2, compute_uv=False).sum(),
    )

N = 3
difference between exact e^tL and 10000 steps 1st order update:
 1.274859357841385e-05
difference between exact e^tL and 100000 steps 1st order update:
 1.2748936328679678e-06
N = 4
difference between exact e^tL and 10000 steps 1st order update:
 1.718268847795462e-05
difference between exact e^tL and 100000 steps 1st order update:
 1.718414337435856e-06
N = 5
difference between exact e^tL and 10000 steps 1st order update:
 2.1307887020738942e-05
difference between exact e^tL and 100000 steps 1st order update:
 2.132984417422713e-06


In [ ]:
'''
Compare the Trotter evolution with different steps for e^{tD_j} = 1 + t D_j
'''
import numpy as np
import qutip as qt

gamma = 0.1 # 1.0 没存 100 步差分

# with the same rho_0
# 这地方后来发现其实有错，return rho 的缩进错误到了单 r 里面，但可以验证 10000步没有明显提升
# different steps for implement e^{tD_j} = 1 + t D_j

for N in [3, 4, 5]:
    r = 3
    gamma = 0.1
    step = 10000
    
    rho_Trotter_exact = np.load(
        f"./data_sigmam/|000>initial/gamma_{gamma}/Trotter_Euler_10000/rho_N_{N}_r_{r}_no_Euler.npy"
    )
    
    rho_Trotter_Euler_10000 = np.load(
        f"./data_sigmam/|000>initial/gamma_{gamma}/Trotter_Euler_10000/rho_N_{N}_r_{r}.npy"
    )

    rho_Trotter_Euler_1000 = np.load(
        f"./data_sigmam/|000>initial/gamma_{gamma}/Trotter_Euler_1000/rho_N_{N}_r_{r}.npy"
    )
    diff1 = rho_Trotter_Euler_1000 - rho_Trotter_exact
    diff2 = rho_Trotter_Euler_10000 - rho_Trotter_exact
    
    print(f'N = {N}, r = {r}:')
    print(
        "difference of Trotter evolution between 1000 steps Euler and exact:\n",
        np.linalg.svd(diff1, compute_uv=False).sum(),
    )
    if 1:
        print(
            "difference of Trotter evolution between 10000 steps Euler and exact:\n",
            np.linalg.svd(diff2, compute_uv=False).sum(),
        )

N = 3, r = 3:
difference of Trotter evolution between 1000 steps Euler and exact:
 1.8684421259490886e-07
difference of Trotter evolution between 10000 steps Euler and exact:
 1.832319539522984e-08
N = 4, r = 3:
difference of Trotter evolution between 1000 steps Euler and exact:
 2.4413268705555854e-07
difference of Trotter evolution between 10000 steps Euler and exact:
 2.3941350178153556e-08
N = 5, r = 3:
difference of Trotter evolution between 1000 steps Euler and exact:
 2.9908678848594964e-07
difference of Trotter evolution between 10000 steps Euler and exact:
 2.9330536531112783e-08


In [12]:
import cvxpy as cp
import numpy as np

# 创建一个极小的测试问题
x = cp.Variable()
prob = cp.Problem(cp.Minimize(x), [x >= 0])

try:
    # 强制指定 MOSEK，cvxpy 通常会直接报错，而不是回退
    prob.solve(solver="MOSEK")
    print("竟然成功了？！")
except Exception as e:
    print(f"验证完毕，直接调用确实会报错：\n{e}")

N = 6
gamma = 0.1

H_x, H_z, c_ops = build_dissipative_tfim(N, J=1.0, h=0.5, gamma=gamma)
t = 0.2

L = qt.liouvillian(H_x + H_z, c_ops)
exact = (L * t).expm()

# N = 6 MOSEK 优化器跑40秒还能算出来
print(qt.dnorm(exact, solver="MOSEK"))

竟然成功了？！
1.0


In [ ]:
'''different ways to compute the propagator'''
i = 4
L_i = qt.tensor([qt.qeye(2)] * i + [qt.sigmam()] + [qt.qeye(2)] * (4 - i - 1))
D_i = qt.lindblad_dissipator(L_i)
print("different ways to compute the propagator:")
(D_i * (0.2 / 2)).expm() - qt.propagator(D_i, 0.2 / 2)

different ways to compute the propagator:


Quantum object: dims=[[[2, 2, 2, 2, 2], [2, 2, 2, 2, 2]], [[2, 2, 2, 2, 2], [2, 2, 2, 2, 2]]], shape=(1024, 1024), type='super', dtype=Dense, isherm=False
Qobj data =
[[-3.51553869e-07  0.00000000e+00  0.00000000e+00 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00 -2.25716034e-08  0.00000000e+00 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00 -3.51553869e-07 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 ...
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00 ...  0.00000000e+00
  -2.25716034e-08  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]]

In [ ]:
# compare the Trotter output with the exact and superexact
import numpy as np
import qutip as qt

t = 0.2
N = 5
gamma = 0.1
r = 2

rho_Trotter = np.load(f"./data_sigmam/|000>initial/gamma_{gamma}/Trotter_Euler_1000/rho_N_{N}_r_{r}.npy")
rho_exact = np.load(f"./data_sigmam/|000>initial/gamma_{gamma}/exp(tL)_Euler/rho_exact_N_{N}_step_10000.npy")
rho_superexact = np.load(f"./data_sigmam/|000>initial/gamma_{gamma}/superexact/rho_superexact_N_{N}.npy")

# print(np.linalg.svd(rho_Trotter - rho_exact, compute_uv=False).sum())
print("exact trace error:", qt.Qobj(rho_Trotter - rho_exact).norm("tr"))
print("superexact trace error:", qt.Qobj(rho_Trotter - rho_superexact).norm("tr"))

# Observable \sum Z_i
si_z = [
    qt.tensor([qt.qeye(2)] * i + [qt.sigmaz()] + [qt.qeye(2)] * (N - i - 1))
    for i in range(N)
]

O = sum([si_z[i] for i in range(N)]).full()

observable_exact = np.trace(O @ rho_exact)
observable_list = []
observable_list.append(np.trace(O @ rho_Trotter))

exact trace error: 0.00457997080605487
superexact trace error: 0.004570098867270783
